# GPU-Based K-Means Clustering Using CUDA — Problem 5

Lloyd's K-Means on an NVIDIA GPU in CUDA C++: given $N$ points in $\mathbb{R}^D$ and $K$ initial
centroids, alternate between assigning every point to its nearest centroid by squared Euclidean
distance and recomputing every centroid as the mean of its assigned points, until assignments stop
changing or the largest centroid movement is $\le \varepsilon$, or until the iteration limit is
reached. The program reports final assignments, final centroids, the iteration count and the
execution time.

The two centroid-accumulation strategies required by the assignment — **Approach A (global
atomics)** and **Approach B (deterministic two-stage reduction)** — are both implemented and
selectable at run time, and the assignment kernel exists in a baseline and a shared-memory variant.

> **This notebook is generated.** `tools/build_notebook.py` assembles it from the files under
> `src/`, `tests/` and `tools/cells/`, and its `%%writefile` cells are checked to be byte-identical
> to those files (DEC-011). Never edit the notebook by hand: change the repository files and
> regenerate, or the next regeneration silently discards the edit.

## Design decisions (requirements.md §3.1)

| Decision | Selected policy |
|---|---|
| Centroid accumulation (A) | Element-wise global `atomicAdd` on binary64 sums and `uint32` counts (§6) |
| Centroid accumulation (B) | Two-stage ownership-based reduction: chunk-privatised partial sums, then a fixed-order tree combine (DEC-001) — race-free without atomics, and deterministic |
| Convergence | Assignment stability **or** $\Delta \le \varepsilon$ (Euclidean, inclusive); stability takes reporting precedence (DEC-002) |
| Empty clusters | Retain the previous centroid; excluded from $\Delta$ (DEC-003) |
| Final consistency | Report the last $(A^{(t)}, C^{(t)})$ pair; P1 holds by construction, P2 verified and reported as a flag (DEC-004) |
| Tie-breaking | Lowest centroid index on exact binary32 equality, via strict-less-than replacement in ascending $k$ (DEC-005) |
| Iteration counting | Number of completed centroid updates (DEC-006) |
| Numeric types | binary32 points, centroids and distances; binary64 centroid sums, movement and tolerance; `uint32` counts (DEC-007) |
| Assignment mapping | One thread per point, grid-stride, ascending-$j$ distance accumulation (DEC-008) |
| Device layout | Two read-only copies of $X$: dimension-major for assignment and Approach A, row-major for Approach B Stage 1 (DEC-009) |
| Global synchronisation | Kernel boundaries on a single stream only; no cooperative-groups grid sync (DEC-010) |
| Deliverable | This single notebook, generated from `src/` (DEC-011) |
| Workflow | Edit and test locally, push to GitHub, run on a Colab GPU, save the executed notebook under `results/` (DEC-012) |

## How to run

1. **Runtime → Change runtime type → GPU** (ENV-11). The configuration cell fails fast with no GPU.
2. Run the cells top to bottom. The environment cell records the machine (§1.6.2), the
   configuration cell derives `SM_ARCH` from the recorded compute capability, and the smoke test
   proves the toolchain before any program source is compiled.
3. Save the executed notebook with **File → Save a copy in GitHub** to
   `GPU-Based-K-Means-Clustering/results/kmeans_cuda_<YYYYMMDD-HHMM>.ipynb` (DEC-012). GPU results
   count as evidence only from an executed notebook committed there.

## 2. Environment record

Records the machine-dependent items left open by requirements.md §1.6.2 — ENV-01 (GPU model and
compute capability), ENV-02 (CUDA toolkit, driver, `compute-sanitizer`), ENV-03 (host OS and host
compiler) and ENV-08 (profiler availability) — into the `ENV` dictionary that every later cell
reads, and that PERF-008 requires in each benchmark CSV row.

No version is assumed anywhere: whatever `nvidia-smi`, `nvcc` and `g++` report is what is recorded
and printed. A missing tool is recorded as unavailable rather than raising, because ENV-08 allows
the profilers to be absent (PERF-009 and TEST-P03 then fall back to CUDA-event phase timing, which
is reported as a limitation). The configuration cell, not this one, decides whether the recorded
environment is usable.

`nvidia-smi` does not report the SM count, and ENV-06 (FP64:FP32 throughput ratio) is a vendor
specification figure, not a queryable property. The SM count comes from
`cudaGetDeviceProperties` — printed by the smoke test below and required of the program itself by
CUDA-001 — and ENV-06 is cited from the vendor specification in the evaluation section.

In [1]:
# Environment record — requirements.md §1.6.2 (ENV-01..ENV-03, ENV-08), PERF-008.
# Every probe is tolerant: a missing tool is recorded as unavailable, never raised. Nothing here
# assumes a GPU model, a toolkit version or a driver version; the reported strings are the record.
import json
import platform
import shutil
import subprocess
import sys


def _run(cmd, timeout=120):
    """Run a probe command. Returns (ok, text); never raises."""
    if shutil.which(cmd[0]) is None:
        return False, "not installed"
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except Exception as exc:  # a probe must not break the notebook
        return False, f"probe failed: {exc!r}"
    out = (p.stdout or "").strip() or (p.stderr or "").strip()
    return p.returncode == 0, out


def _first_line(text):
    return text.splitlines()[0].strip() if text else ""


def _version_of(cmd):
    """Version string of a tool invoked as `<tool> --version`, or None."""
    ok, out = _run([cmd, "--version"])
    return _first_line(out) if ok else None


ENV = {}

# ENV-03 — host OS and host compiler of the runtime VM.
ENV["os"] = {
    "platform": platform.platform(),
    "release": _run(["cat", "/etc/os-release"])[1].replace("\n", "; ") or None,
    "python": sys.version.split()[0],
}
ENV["host_compiler"] = _version_of("g++")

# ENV-01 — GPU model and compute capability.
gpu_ok, gpu_csv = _run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,driver_version,memory.total",
     "--format=csv,noheader"]
)
ENV["gpu"] = None
if gpu_ok and gpu_csv:
    fields = [f.strip() for f in gpu_csv.splitlines()[0].split(",")]
    if len(fields) == 4:
        ENV["gpu"] = {
            "name": fields[0],
            "compute_cap": fields[1],
            "driver_version": fields[2],
            "memory_total": fields[3],
        }
        ENV["gpu_count"] = len(gpu_csv.splitlines())
if ENV["gpu"] is None:
    ENV["gpu_probe"] = gpu_csv or "nvidia-smi unavailable"

# ENV-02 — CUDA toolkit and the sanitizer the §16.4 suite needs.
nvcc_ok, nvcc_out = _run(["nvcc", "--version"])
ENV["nvcc"] = nvcc_out.replace("\n", " | ") if nvcc_ok else None
ENV["compute_sanitizer"] = _version_of("compute-sanitizer")

# ENV-08 — profiler availability; absence is a recorded limitation, not an error.
ENV["profilers"] = {"nsys": _version_of("nsys"), "ncu": _version_of("ncu")}

# Notebook analysis stack (§1.5 deliverables 4 and 5; no other dependency is permitted).
ENV["python_packages"] = {}
for _pkg in ("numpy", "pandas", "matplotlib"):
    try:
        ENV["python_packages"][_pkg] = __import__(_pkg).__version__
    except Exception:
        ENV["python_packages"][_pkg] = None

print(json.dumps(ENV, indent=2))

{
  "os": {
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "release": "PRETTY_NAME=\"Ubuntu 24.04.4 LTS\"; NAME=\"Ubuntu\"; VERSION_ID=\"24.04\"; VERSION=\"24.04.4 LTS (Noble Numbat)\"; VERSION_CODENAME=noble; ID=ubuntu; ID_LIKE=debian; HOME_URL=\"https://www.ubuntu.com/\"; SUPPORT_URL=\"https://help.ubuntu.com/\"; BUG_REPORT_URL=\"https://bugs.launchpad.net/ubuntu/\"; PRIVACY_POLICY_URL=\"https://www.ubuntu.com/legal/terms-and-policies/privacy-policy\"; UBUNTU_CODENAME=noble; LOGO=ubuntu-logo",
    "python": "3.13.15"
  },
  "host_compiler": "g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0",
  "gpu": {
    "name": "Tesla T4",
    "compute_cap": "7.5",
    "driver_version": "580.82.07",
    "memory_total": "15360 MiB"
  },
  "gpu_count": 1,
  "nvcc": "nvcc: NVIDIA (R) Cuda compiler driver | Copyright (c) 2005-2025 NVIDIA Corporation | Built on Wed_Aug_20_01:58:59_PM_PDT_2025 | Cuda compilation tools, release 13.0, V13.0.88 | Build cuda_13.0.r13.0/compiler.36424714_0",
  "compu

## 3. Configuration

Derives everything machine-dependent from the record above — nothing is hard-coded before ENV-01
is known (BLD-001c):

- **`SM_ARCH`** from the reported compute capability (`7.5` → `sm_75`).
- **Fail fast** when the environment cannot run the assignment: no GPU attached, compute capability
  below 6.0 (CUDA-001 — `atomicAdd` on `double`, which ATOM-003 requires, is not native below
  6.0), or a CUDA 13.x toolkit paired with a device below CC 7.5, whose offline compilation that
  toolkit removed (BLD-001a). Each case raises with the action to take, so "restart kernel and run
  all" stops here rather than failing later inside a build.
- **Build flags** (BLD-002): a release build `-O3 -lineinfo -arch=$SM_ARCH -std=c++17` with
  `-Xptxas -v` so register and shared-memory usage appear in the build log (CUDA-010), and a
  sanitizer/debug build that adds `-DKMEANS_SYNC_CHECK` (CUDA-008). `--use_fast_math` is never
  used (NUM-002), and the CPU reference is compiled with FP contraction disabled (NUM-003).
- **Paths**: results — benchmark CSVs, output files and logs — go to Google Drive when
  `USE_DRIVE` is enabled and the mount succeeds, so a runtime disconnect does not lose them;
  otherwise a warning is printed, local disk is used, and the fallback is recorded in `ENV` so
  results that were not persisted stay identifiable (BLD-001c). Sources, binaries and generated
  datasets always stay on local disk: the Drive FUSE mount does not permit executing binaries, so
  anything built there could not be run, and datasets are reproducible from their seeds (FR-003)
  while reaching tens of GB at the top of the PERF-003 matrix. The cell `chdir`s into the build
  directory, which is why the `%%writefile` cells use flat file names.

In [2]:
# Configuration — requirements.md §15.3 part 3; BLD-001a, BLD-001c, BLD-002, CUDA-001, NUM-002/003.
import os
import re
import subprocess

USE_DRIVE = True  # BLD-001c: persist results to Drive; falls back to local disk on failure.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/kmeans_cuda"
BUILD_DIR = "/content/kmeans"  # always local: see the note on exec permissions below

if "ENV" not in globals():
    raise RuntimeError("Run the environment cell (section 2) first: ENV is not defined.")

# --- ENV-01 -> SM_ARCH (BLD-001c: derived, never hard-coded) ------------------------------------
if not ENV.get("gpu"):
    raise RuntimeError(
        "No NVIDIA GPU visible (ENV-01). Set Runtime -> Change runtime type -> Hardware "
        "accelerator: GPU, then re-run from section 2. Probe said: "
        f"{ENV.get('gpu_probe', 'unknown')}"
    )

COMPUTE_CAP = ENV["gpu"]["compute_cap"].strip()
_cc_match = re.fullmatch(r"(\d+)\.(\d+)", COMPUTE_CAP)
if not _cc_match:
    raise RuntimeError(f"Could not parse compute capability {COMPUTE_CAP!r} reported by nvidia-smi.")
CC_MAJOR, CC_MINOR = int(_cc_match.group(1)), int(_cc_match.group(2))
CC_VALUE = CC_MAJOR + CC_MINOR / 10.0
SM_ARCH = f"sm_{CC_MAJOR}{CC_MINOR}"

# CUDA-001: ATOM-003 needs native atomicAdd(double), which arrived with compute capability 6.0.
if CC_VALUE < 6.0:
    raise RuntimeError(
        f"Compute capability {COMPUTE_CAP} is below the 6.0 minimum (CUDA-001): native "
        "atomicAdd(double) required by ATOM-003 is unavailable. Use a newer GPU runtime."
    )

# ENV-02: whatever nvcc reports is what is used; no toolkit version is assumed.
NVCC_VERSION = ENV.get("nvcc")
if not NVCC_VERSION:
    raise RuntimeError("nvcc is not available in this runtime (ENV-02); CUDA code cannot be built.")
_rel = re.search(r"release (\d+)\.(\d+)", NVCC_VERSION)
CUDA_TOOLKIT = f"{_rel.group(1)}.{_rel.group(2)}" if _rel else "unknown"

# BLD-001a: CUDA 13.x dropped offline compilation for compute capability < 7.5.
if _rel and int(_rel.group(1)) >= 13 and CC_VALUE < 7.5:
    raise RuntimeError(
        f"CUDA toolkit {CUDA_TOOLKIT} cannot compile offline for compute capability "
        f"{COMPUTE_CAP} (BLD-001a). Use a CUDA 12.x toolkit, or a GPU with CC >= 7.5."
    )

# --- BLD-002 build flags ------------------------------------------------------------------------
# No --use_fast_math anywhere (NUM-002): it would change the distance semantics of §2.2.
NVCC_COMMON = ["-std=c++17", f"-arch={SM_ARCH}", "-lineinfo"]
NVCC_RELEASE_FLAGS = NVCC_COMMON + ["-O3", "-Xptxas", "-v"]          # CUDA-010 usage in build log
NVCC_DEBUG_FLAGS = NVCC_COMMON + ["-O3", "-DKMEANS_SYNC_CHECK"]      # CUDA-008 localises faults
# NUM-003: the CPU reference is the oracle and must not contract multiply-adds.
NVCC_HOST_FP_FLAGS = ["-Xcompiler", "-ffp-contract=off"]
GXX_REFERENCE_FLAGS = ["-O2", "-std=c++17", "-ffp-contract=off"]

# --- BLD-001c paths ----------------------------------------------------------------------------
# Sources, binaries and generated datasets stay on local disk and only results go to Drive, for
# two reasons: the Drive FUSE mount does not permit executing binaries, so a kmeans built there
# could not be run, and datasets reach tens of GB at the top of the PERF-003 matrix while being
# reproducible from their seeds (FR-003), so they are not worth persisting.
RESULTS_ROOT, DRIVE_STATUS = BUILD_DIR, "disabled"
if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        RESULTS_ROOT, DRIVE_STATUS = DRIVE_RESULTS_DIR, "mounted"
    except Exception as exc:
        DRIVE_STATUS = f"unavailable: {exc!r}"
        print(
            "warning[BLD-001c]: Google Drive not mounted; results stay in "
            f"{BUILD_DIR} and are lost if the runtime disconnects, so download anything you "
            f"need before closing the session. Reason: {exc!r}"
        )

WORK_DIR = BUILD_DIR                              # cwd for the %%writefile cells and the compiler
DATA_DIR = os.path.join(BUILD_DIR, "data")        # generated datasets (FR-003), regenerable
OUT_DIR = os.path.join(RESULTS_ROOT, "out")       # .kma/.kmd outputs and JSON summaries (BLD-004)
CSV_DIR = os.path.join(RESULTS_ROOT, "csv")       # benchmark CSVs (section 13.1)
LOG_DIR = os.path.join(RESULTS_ROOT, "logs")      # test and sanitizer logs (section 1.5)
for _d in (BUILD_DIR, DATA_DIR, OUT_DIR, CSV_DIR, LOG_DIR):
    os.makedirs(_d, exist_ok=True)
os.chdir(WORK_DIR)  # the %%writefile cells below use flat names relative to this directory

# PERF-008 asks for a build identity per benchmark row. In Colab there is no git checkout, so the
# executed notebook saved under results/ is itself the evidence (DEC-012); record what is knowable.
try:
    RUN_VERSION = "git:" + subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=30
    ).stdout.strip()
except Exception:
    RUN_VERSION = "unknown"
if RUN_VERSION == "git:":
    RUN_VERSION = "unknown"

ENV["drive_status"] = DRIVE_STATUS          # BLD-001c: unpersisted results stay identifiable
ENV["work_dir"] = WORK_DIR
ENV["results_root"] = RESULTS_ROOT
ENV["sm_arch"] = SM_ARCH
ENV["cuda_toolkit"] = CUDA_TOOLKIT
ENV["run_version"] = RUN_VERSION

print(f"GPU             : {ENV['gpu']['name']} (compute capability {COMPUTE_CAP})")
print(f"SM_ARCH         : {SM_ARCH}   [derived from ENV-01, BLD-001c]")
print(f"CUDA toolkit    : {CUDA_TOOLKIT} (as reported by nvcc)")
print(f"Drive           : {DRIVE_STATUS}")
print(f"build dir (cwd) : {WORK_DIR}   [local: Drive mounts are not executable]")
print(f"results root    : {RESULTS_ROOT}")
print(f"data / out / csv: {DATA_DIR} | {OUT_DIR} | {CSV_DIR}")
print(f"run version     : {RUN_VERSION}")
print(f"nvcc release    : {' '.join(NVCC_RELEASE_FLAGS)}")
print(f"nvcc debug      : {' '.join(NVCC_DEBUG_FLAGS)}")
print(f"host FP (oracle): {' '.join(NVCC_HOST_FP_FLAGS)} / g++ {' '.join(GXX_REFERENCE_FLAGS)}")

Mounted at /content/drive
GPU             : Tesla T4 (compute capability 7.5)
SM_ARCH         : sm_75   [derived from ENV-01, BLD-001c]
CUDA toolkit    : 13.0 (as reported by nvcc)
Drive           : mounted
build dir (cwd) : /content/kmeans   [local: Drive mounts are not executable]
results root    : /content/drive/MyDrive/kmeans_cuda
data / out / csv: /content/kmeans/data | /content/drive/MyDrive/kmeans_cuda/out | /content/drive/MyDrive/kmeans_cuda/csv
run version     : unknown
nvcc release    : -std=c++17 -arch=sm_75 -lineinfo -O3 -Xptxas -v
nvcc debug      : -std=c++17 -arch=sm_75 -lineinfo -O3 -DKMEANS_SYNC_CHECK
host FP (oracle): -Xcompiler -ffp-contract=off / g++ -O2 -std=c++17 -ffp-contract=off


## 4. Toolchain smoke test

A nine-line kernel is cheaper to debug than a multi-file build, so before any program source is
compiled this section writes `hello.cu`, builds it with the **same release flags the real build
uses** (BLD-002, so a bad flag is caught here), runs it, and checks three things:

1. `nvcc` accepts `-arch=$SM_ARCH` for the attached device and the binary links and runs;
2. a kernel launches and its writes are visible to the host (`kernel.check=OK`, verified
   element-wise, not just by a checksum);
3. the compute capability the device reports to host code equals the one `nvidia-smi` reported in
   section 2 — if these disagree, `SM_ARCH` was derived from the wrong device.

The cell raises on a non-zero compile or run exit code, on a mismatch, or on a missing `OK`, so
"restart kernel and run all" halts at the first toolchain fault (NFR-014). It also prints the SM
count, shared memory per block, warp size and the CUDA runtime and driver versions — the
`cudaGetDeviceProperties` fields that `nvidia-smi` does not expose, which CUDA-001 requires the
program to query, MEM-004 and SMEM-003 check tile sizes against, and CUDA-007 uses to size grids.
Whatever the runtime reports is printed verbatim; no CUDA version is assumed.

In [3]:
%%writefile hello.cu
// Toolchain smoke test (requirements.md §15.3 part 4).
//
// Proves, before any program source is compiled, that the SM_ARCH derived from ENV-01 matches the
// attached device, that the toolkit reported by nvcc can build and link for it, and that a kernel
// launches and its writes are visible to the host. It also prints the device properties that
// nvidia-smi does not expose (SM count, shared memory per block) and that CUDA-001 requires the
// program itself to query, plus the CUDA runtime and driver versions exactly as reported — no
// version is assumed anywhere.
//
// Output is `key=value` lines so the notebook cell can assert on them. Exit codes follow BLD-005:
// 0 success, 4 CUDA runtime error.

#include <cstdio>
#include <cstdlib>

#include <cuda_runtime.h>

// CUDA-008: every runtime call is checked; failure reports file, line, call and the error string.
#define CUDA_CHECK(call)                                                                     \
    do {                                                                                     \
        cudaError_t err_ = (call);                                                            \
        if (err_ != cudaSuccess) {                                                            \
            std::fprintf(stderr, "error[CUDA-008]: %s:%d: %s failed: %s\n", __FILE__,          \
                         __LINE__, #call, cudaGetErrorString(err_));                           \
            std::exit(4);                                                                      \
        }                                                                                     \
    } while (0)

static const int kElements = 256;

// One thread per element, grid-stride (CUDA-007), bounds-checked (CUDA-005). The value is i*i so
// the host can verify every element independently of launch geometry.
__global__ void square_indices(int *out, int n)
{
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x) {
        out[i] = i * i;
    }
}

int main()
{
    int device = 0;
    CUDA_CHECK(cudaGetDevice(&device));

    cudaDeviceProp prop{};
    CUDA_CHECK(cudaGetDeviceProperties(&prop, device));

    int runtime_version = 0;
    int driver_version = 0;
    CUDA_CHECK(cudaRuntimeGetVersion(&runtime_version));
    CUDA_CHECK(cudaDriverGetVersion(&driver_version));

    std::printf("device.name=%s\n", prop.name);
    std::printf("device.compute_capability=%d.%d\n", prop.major, prop.minor);
    std::printf("device.sm_count=%d\n", prop.multiProcessorCount);
    std::printf("device.global_mem_bytes=%zu\n", prop.totalGlobalMem);
    std::printf("device.shared_mem_per_block_bytes=%zu\n", prop.sharedMemPerBlock);
    std::printf("device.shared_mem_per_block_optin_bytes=%d\n", prop.sharedMemPerBlockOptin);
    std::printf("device.warp_size=%d\n", prop.warpSize);
    std::printf("device.max_threads_per_block=%d\n", prop.maxThreadsPerBlock);
    std::printf("runtime.cuda_version=%d\n", runtime_version);
    std::printf("driver.cuda_version=%d\n", driver_version);

    int *d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_out, kElements * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_out, 0, kElements * sizeof(int)));

    square_indices<<<4, 64>>>(d_out, kElements);
    CUDA_CHECK(cudaGetLastError());  // launch error (CUDA-008)

    int host_out[kElements];
    CUDA_CHECK(cudaMemcpy(host_out, d_out, sizeof(host_out), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));

    long long sum = 0;
    long long expected_sum = 0;
    int mismatches = 0;
    for (int i = 0; i < kElements; ++i) {
        sum += host_out[i];
        expected_sum += static_cast<long long>(i) * i;
        if (host_out[i] != i * i) {
            ++mismatches;
        }
    }

    std::printf("kernel.elements=%d\n", kElements);
    std::printf("kernel.sum=%lld\n", sum);
    std::printf("kernel.expected_sum=%lld\n", expected_sum);
    std::printf("kernel.mismatches=%d\n", mismatches);
    std::printf("kernel.check=%s\n", (mismatches == 0 && sum == expected_sum) ? "OK" : "FAIL");

    return (mismatches == 0 && sum == expected_sum) ? 0 : 1;
}


Writing hello.cu


In [4]:
# Toolchain smoke test — requirements.md §15.3 part 4. Raises on any failure so that
# "restart kernel and run all" stops here instead of inside a larger build (NFR-014).
import subprocess

if "SM_ARCH" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first: SM_ARCH is not defined.")

compile_cmd = ["nvcc"] + NVCC_RELEASE_FLAGS + ["-o", "hello", "hello.cu"]
print("$ " + " ".join(compile_cmd))
compile_proc = subprocess.run(compile_cmd, capture_output=True, text=True)
print(compile_proc.stdout + compile_proc.stderr, end="")  # -Xptxas -v usage report (CUDA-010)
if compile_proc.returncode != 0:
    raise RuntimeError(
        f"nvcc failed with exit code {compile_proc.returncode} for -arch={SM_ARCH} "
        f"(toolkit {CUDA_TOOLKIT}, device CC {COMPUTE_CAP}). See the output above; check BLD-001a "
        "if the toolkit is too new for this device."
    )

run_proc = subprocess.run(["./hello"], capture_output=True, text=True)
print(run_proc.stdout + run_proc.stderr, end="")
if run_proc.returncode != 0:
    raise RuntimeError(
        f"./hello exited with code {run_proc.returncode} "
        "(4 = CUDA runtime error per BLD-005). See the output above."
    )

smoke = dict(
    line.split("=", 1) for line in run_proc.stdout.splitlines() if "=" in line
)
if smoke.get("kernel.check") != "OK":
    raise RuntimeError(
        f"Kernel result wrong: {smoke.get('kernel.mismatches')} of "
        f"{smoke.get('kernel.elements')} elements mismatched. The toolchain builds but the device "
        "does not compute correctly; do not trust any later result."
    )
if smoke.get("device.compute_capability") != COMPUTE_CAP:
    raise RuntimeError(
        f"Device reports compute capability {smoke.get('device.compute_capability')} to host code "
        f"but nvidia-smi reported {COMPUTE_CAP} in section 2, so SM_ARCH={SM_ARCH} was derived "
        "from a different device (BLD-001c). Re-run from section 2."
    )

# SM count feeds CUDA-007 grid sizing and the Approach B chunk count G (RED-006); the shared-memory
# limits bound the assign_smem tiles (SMEM-003, MEM-004). Recorded now, used from M2 on.
ENV["device_properties"] = smoke
SM_COUNT = int(smoke["device.sm_count"])
SHARED_MEM_PER_BLOCK = int(smoke["device.shared_mem_per_block_bytes"])

print(
    f"\nsmoke test OK: {smoke['device.name']}, CC {smoke['device.compute_capability']}, "
    f"{SM_COUNT} SMs, {SHARED_MEM_PER_BLOCK} B shared memory per block, "
    f"CUDA runtime {smoke['runtime.cuda_version']}, driver {smoke['driver.cuda_version']}, "
    f"kernel sum {smoke['kernel.sum']} == expected {smoke['kernel.expected_sum']}"
)

$ nvcc -std=c++17 -arch=sm_75 -lineinfo -O3 -Xptxas -v -o hello hello.cu
ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z14square_indicesPii' for 'sm_75'
ptxas info    : Function properties for _Z14square_indicesPii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 12 registers, used 0 barriers, 364 bytes cmem[0]
ptxas info    : Compile time = 73.823 ms
hello.cu: In function ‘int main()’:
hello.cu:58:13: warning: format ‘%d’ expects argument of type ‘int’, but argument 2 has type ‘size_t’ {aka ‘long unsigned int’} [-Wformat=]
   58 |     std::printf("device.shared_mem_per_block_optin_bytes=%d\n", prop.sharedMemPerBlockOptin);
      |             ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
      |                                                                  |
      |                                                                  size_t {aka long unsigned int}
device.name=Tesla T4
device.